[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/closed_universe/hopf_projection.ipynb)

# Gravitational Waves Below the Hopf Circles

This example projects the gravitational waves of a closed universe, the round three-sphere, onto the ordinary sphere below its Hopf circles, and watches a quadrupole travel round it. The three-sphere is a bundle of great circles, its Hopf circles, each as far from every other all along it, and each circle stands over one point of an ordinary sphere, the base; the 600-cell's twelve guardrail decagons are such circles, over the corners of an icosahedron. A wave that is the same all along the circles loses nothing on the base: read along a circle it says how much the circle stretches, which colours the base, and read across the circles it is the strain of the base itself, which small triangles of atoms on the base feel and a bundle of light crossing it bends to. The waves here are the 600-cell's squash and the squash turned an eighth of a turn, a quarter swing behind it, which together turn round the base.

Points of the three-sphere are antivectors, and the circles are the orbits of a turn acting on the same side as the isoclinic frames, `isoclinic(place)`. Read in its frame, a circle's direction, `isoclinic(place).inverse() >> place.commutator(circling)`, is the same step of the origin all along the circle: the base is the sphere of such steps, and that step is the circle's base point. Steps across the circles are the base's own directions turned a quarter about the base point, `turning(base).commutator(direction)`, and the Hopf map doubles their angles. Lifting the base's triangles across the circles turns them into triangles of the three-sphere, which roll through the waves on springs; holding each atom to its place by a soft spring keeps a triangle from taking up the turning wave's angular momentum, so that only its shape responds to the wave.

In quaternion notation, the Hopf map sends a unit quaternion $q$ to the unit imaginary quaternion $\bar q\,\mathbf{i}\,q$, the same all along the circle $e^{\mathbf{i}t} q$; the base is the sphere of unit imaginary quaternions, whose angles across the circles are twice the three-sphere's, so that two circles a quarter turn apart stand over antipodes of the base.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np
from numga import NumpyContext, stack

from examples.util.animation import show_animation
from examples.relativity.closed_universe import core, render, scenarios
from examples.util.topology import Topology

np.set_printoptions(precision=4, suppress=True)

ga = core.ga
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                     # Scalar+
# Planes through the origin, the polar planes of its steps; the rates that move and turn frames.
Plane = ga.gatype.vector()                                                      # Vector+
Through = Plane.derive.within("x y z")                                          # Vector+
Rate = ga.gatype.bivector()                                                     # Bivector+
# Points of the sphere and the origin's steps, the base's points among them; the motors that carry them.
Point = ga.gatype.antivector()                                                  # Antivector-
Step = Point.derive.containing("w")                                             # Antivector-
Motor = ga.gatype.rotor()                                                       # Rotor+
Translating = ga.gatype((Step, Through))                                        # Antivector- <- Vector+

# The origin's equator, the plane `w`, and the origin, its pole; the extrinsic pseudoscalar; each step's polar plane.
equator = mv.w                                                                  # [] Plane+
origin = equator.dual()                                                         # [] Point-
pseudoscalar = mv.xyzw                                                          # [] Antiscalar+
polar = origin.commutator(equator ^ Through).cast(Translating).inverse()        # [] Vector+ <- Antivector-


def isoclinic(places: Point) -> Motor:
    """The isoclinic frames at places on the sphere, `Motor+`: the motors that carry the origin to each by turning
    alike in the plane of the great circle between them and in the plane square to it."""
    # The half of the quotient of the place by the origin that the pseudoscalar keeps, with one in the other half.
    return ((1 + pseudoscalar) * (places.normalized() / origin) + (1 - pseudoscalar)) / 2

## The base below the circles

The 600-cell's decagons are the orbits of the turn that carries each vertex a tenth of the way along its circle, a turn that carries the whole 600-cell onto itself. A place's base point is its circle's direction read in its frame. A point on the circle over a base point is the origin moved square to its own base point and to that one, by half the angle between them, which the Hopf map doubles.

In [ ]:
cell_mesh, cell_vertices = scenarios.six_hundred_cell()
turn = scenarios.circle_turn(cell_vertices, scenarios.guardrail_sides)          # [] Motor+
circling = turn.log()                                                           # [] Rate+


def base_points(places: Point) -> Point:
    """The base point below each place, `Point-`: the direction its circle runs in, read in the isoclinic frame at
    the place, a unit step of the origin."""
    along = places.commutator(circling)                                        # Point-
    return (isoclinic(places).inverse() >> along).cast(Step).normalized()


def lifts(bases: Point) -> Point:
    """A point on the circle over each base point, `Point-`."""
    below_origin = base_points(origin)                                          # [] Point-
    # The plane the two base points turn in, times the pseudoscalar: the rate that moves the origin square to both.
    lifting = ((polar(bases) ^ polar(below_origin)) * pseudoscalar).normalized()   # [] Rate+
    angles = (polar(bases) & below_origin).arccos()                            # Scalar+
    return (lifting * (-angles / 4)).exp() >> origin


# The decagons' base points, the corners of an icosahedron.
decagons = scenarios.guardrails(cell_vertices, scenarios.guardrail_sides)        # [decagons, sides]
decagon_bases = base_points(cell_vertices.batch()[decagons[:, 0]])              # [decagons] Point-

## The quadrupole, read along the circles

The waves are the 600-cell's squash, which stretches space along `x` and squeezes it along `y` in the frames, and the squash turned an eighth of a turn about `z`, a quarter swing behind it: the two turn the stretch round about `z`, and on the base the quadrupole travels round. Squashes are the same everywhere in the frames, so they are the same all along every circle. Read along a circle, half the strain's reading of the circle's direction on itself is how much the circle stretches, averaged here over a few points round it, between the tenth turns that carry the 600-cell onto itself. This cell takes a few seconds, most of it stepping the waves.

In [ ]:
# The points sampled round each circle.
n_circle_samples = 7

In [ ]:
waves, frequency = scenarios.quadrupole_history(cell_mesh, cell_vertices)
sphere_mesh, bases = scenarios.base_sphere()
pick = core.picking(cell_mesh, cell_vertices)


def stretches(bases: Point, directions: Point, time: float) -> Scalar:
    """How much lengths stretch along the steps of the origin `directions` in the frames, at the circles over the base
    points, averaged round each, at a time, `Scalar+[bases]`: half the strain's reading of the step on itself."""
    shares = np.arange(n_circle_samples) * scenarios.guardrail_sides / n_circle_samples   # [samples]
    places = (circling * shares[:, None]).exp() >> lifts(bases)[None]           # [samples, bases] Point-
    strain, _, _, _ = waves.fields(pick(places.reshape(-1)), places.reshape(-1), time)   # [samples * bases] Vector+ <- Antivector-
    along = directions[None].broadcast_to(places.shape).reshape(-1)            # [samples * bases] Antivector-
    readings = (strain(along) & along) / (2 * along.dot(along))                 # [samples * bases] Scalar+
    return readings.reshape(places.shape).sum(axis=0) / n_circle_samples

## Triangle gauges

In each face of the icosahedron the decagons' base points make, a triangle of atoms stands with its corners drawn toward the face's middle. Lifted from the circle over the middle across the circles, its bonds run across them too, all but a little: the circles twist about the triangle's area, so no closed triangle runs across them everywhere. The atoms roll through the waves on springs along the bonds, tuned so that the triangle's shear swings at the waves' frequency and damped, and each atom is held to its place, a free particle at rest, by a soft spring damped critically. Without that hold a triangle near the poles of the turning, where the quadrupole turns like a spinning stretch, takes up the wave's angular momentum and spins. This cell takes about ten seconds, most of it rolling the atoms.

In [ ]:
# The corners' share of the way from a face's middle to its corners; the hold's frequency as a share of the waves';
# the shear swing's damping as a share of critical; the points drawn along each bond.
gauge_share = 0.4
holding_share = 1 / 4
gauge_damping = 0.4
n_arc_points = 12
# The frames over one swing, and a frame's duration in milliseconds.
n_frames = 75
duration_ms = 40
# The settling swings before the frames, and the time step of the atoms.
n_settling_swings = scenarios.n_settling_swings
atom_step = scenarios.crystal_step

In [ ]:
# The icosahedron's faces, and in each a triangle with its corners drawn toward the face's middle.
faces = scenarios.hull_faces(decagon_bases)                                     # [faces, 3]
corners = decagon_bases[faces]                                                  # [faces, 3] Point-
middles = corners.sum(axis=-1).normalized()                                     # [faces] Point-
below = (middles[:, None] + gauge_share * (corners - middles[:, None])).normalized()   # [faces, 3] Point-
# Each atom's direction along the base from its face's middle and the cosine of the angle between them: the step
# across the circles that the direction is the image of, a quarter turn about the middle, carries the middle's lift
# half that angle.
cosines = polar(middles[:, None]) & below                                       # [faces, 3] Scalar+
directions = (below - middles[:, None] * cosines).cast(Step).normalized()       # [faces, 3] Point-
across = ((equator ^ polar(middles[:, None])) * pseudoscalar).commutator(directions).cast(Step)   # [faces, 3] Point-
moves = ((equator ^ polar(across)) * (-cosines.arccos() / 4)).exp()             # [faces, 3] Motor+
atoms = ((isoclinic(lifts(middles))[:, None] * moves) >> origin).reshape(-1).field()   # Point-[atoms]
# Each triangle's three sides, its atoms three to a face.
bonds = Topology.loops([3] * len(faces))

In [ ]:
# The hold, and the bonds' springs and dashpots: a triangle of unit masses swings in shear at the root of three halves
# of its bonds' stiffness and the hold, damped by three halves of its bonds' dashpots and by the hold's.
period = 2 * np.pi / frequency
holding = (holding_share * frequency) ** 2
n_bonds = bonds.n_elements[1]
bond_stiffness = mv.scalar(np.full((n_bonds, 1), 2 / 3 * (frequency ** 2 - holding))).field()   # Scalar+[bonds]
bond_damping = mv.scalar(np.full((n_bonds, 1), 2 / 3 * (2 * gauge_damping * frequency - 2 * np.sqrt(holding)))).field()   # Scalar+[bonds]
bond_tails, bond_heads = bonds.incidence[1, 0] * atoms                          # Point-[bonds], Point-[bonds]
bond_steps = (isoclinic(bond_tails).inverse() >> bond_heads).cast(Step)        # Antivector-[bonds]
bond_rest = bond_steps.dot(bond_steps).square_root()                            # Scalar+[bonds]
times = np.arange(n_frames) * period / n_frames                                 # [frames]
at_rest = (mv.x.dual().cast(Step) * np.zeros(bonds.n_vertices)).field()         # Antivector-[atoms]
states = list(core.massive(cell_mesh, cell_vertices, waves, bonds, bond_rest, bond_stiffness, bond_damping, holding,
                           atoms, at_rest, times[0] - n_settling_swings * period, atom_step, times))

## A bundle of light

A narrow bundle of rays sets out across the circles from the circle over the base point along `z`, its headings the base's directions about the one toward `y`, which faces the viewer, each turned a quarter about the base point. Through the round sphere every ray would run along a great circle across the circles and reach the base's far pole after a quarter turn; through the waves the rays sway, and their paths are seen on the base. A fresh bundle sets out at every frame.

In [ ]:
# The rays, the angle they are spread over, and the length of their steps, for a quarter turn.
n_rays = 7
light_spread = np.radians(18)
light_step = 0.02
n_light_steps = int(np.pi / 2 / light_step)

In [ ]:
source, toward = mv.z.dual().cast(Step), mv.y.dual().cast(Step)               # [] Point-, [] Point-
turning_source = (equator ^ polar(source)) * pseudoscalar                       # [] Rate+
aim = (toward - source * (polar(source) & toward)).cast(Step).normalized()      # [] Point-
fan = (turning_source * (-np.linspace(-light_spread / 2, light_spread / 2, n_rays) / 2)).exp()   # [rays] Motor+
headings = turning_source.commutator(fan >> aim).cast(Step).normalized()        # [rays] Point-
starts = lifts(source[None]).broadcast_to((n_rays,))                            # [rays] Point-

## The base from outside

Each frame shows the base from outside: shaded by how much the circles over it stretch, red where they lengthen and blue where they shorten, the decagons' base points in their own colours, the triangles' bonds drawn as the arcs they make on the base and coloured by their strain, and the light's paths. This cell takes about half a minute, most of it tracing the light and the frames.

In [ ]:
def on_base() -> Iterator[tuple]:
    """At each frame, all on the base: the circles' stretch at the icosphere's points, the atoms, the bonds' arcs and
    strains, and the light's paths."""
    shares = np.linspace(0, 1, n_arc_points)                                    # [arc points]
    for time, (positions, strains) in zip(times, states):
        tails, heads = bonds.incidence[1, 0] * positions                       # Point-[bonds], Point-[bonds]
        along_bonds = (tails.batch()[:, None] * (1 - shares) + heads.batch()[:, None] * shares).normalized()   # [bonds, arc points] Point-
        traced = core.light(cell_mesh, cell_vertices, waves, starts, headings, time, light_step, n_light_steps)
        paths = stack([starts, *traced], axis=-1)                               # [rays, steps + 1] Point-
        yield (stretches(bases, bases, time), base_points(positions.batch()), base_points(along_bonds),
               strains.batch(), base_points(paths))


# Each frame drawn as it comes.
show_animation((render.base_view(sphere_mesh, bases, circle_stretch, decagon_bases, atoms_seen, arcs, strains,
                                 light, scenarios.wave_swing)
                for circle_stretch, atoms_seen, arcs, strains, light in on_base()),
               "closed_universe_quadrupole", duration_ms)

In [ ]:
# checks
# The quadrupole is the same all along each circle: a circle's stretch read at one point of it is its average round
# it; the triangles' atoms stand over their corners; and the light sets out across the circles.
places = lifts(bases)                                                           # [points] Point-
strain, _, _, _ = waves.fields(pick(places), places, 0.0)                       # [points] Vector+ <- Antivector-
at_one = (strain(bases) & bases) / (2 * bases.dot(bases))                       # [points] Scalar+
averaged = stretches(bases, bases, 0.0)                                         # [points] Scalar+
assert np.abs((at_one - averaged).to_array()).max() < 1e-6 * np.abs(averaged.to_array()).max()
np.testing.assert_allclose((polar(base_points(atoms.batch())) & below.reshape(-1)).to_array(), 1, atol=1e-12)
np.testing.assert_allclose((polar(base_points(starts)) & headings).to_array(), 0, atol=1e-12)